# Volatility model: how strongly will the price move in the next hour?

**Owner:** Dennis · **Stage 3** · task `volatility`, horizon 4 candles (1 hour), interval 15m

The direction model (notebook 01) asks *which way* the next candle goes. This one asks *how much* the price will
move, whatever the direction. That is the **volatility**: here, the standard deviation of the next four
15-minute log returns (`outcomes.realized_vol`).

Why bother, if it says nothing about up or down?
- It is far more predictable than direction, because volatility **clusters**: calm hours follow calm hours,
  wild hours follow wild hours.
- It is what risk decisions need: how big a position to take, how far away to put a stop.

**Steps:** 1. load the data · 2. look at the target · 3. split by time · 4. compare a baseline and two models ·
5. test the chosen model once · 6. what it learned · 7. save, register, activate · 8. write predictions

The logic lives in `src/models/train_model.py` (`--task volatility`), the same code as for the direction model.

**Before running:** Docker is up, and `extract_rest.py` → `transform.py` → `src/features/build_features.py` have run.
Kernel: the project's `.venv`.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# work from the repo root, whether the notebook is started from the root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "sql").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src" / "models"))

import train_model as tm          # noqa: E402
from db import get_pg_connection  # noqa: E402  (train_model puts src/data on the path)

TASK = "volatility"
CFG = tm.TASKS[TASK]
TARGET = CFG["target"]

pd.set_option("display.float_format", lambda x: f"{x:.5f}")
pd.set_option("display.width", 140)
print("target:", TARGET, "| horizon:", CFG["horizon"], "candles | models:", CFG["models"],
      "| picked by:", CFG["metric"], "| learns log(target):", CFG["log_target"])

target: realized_vol | horizon: 4 candles | models: ('baseline', 'ridge', 'hgb_reg') | picked by: rmse | learns log(target): True


## 1. Load the data

One row per closed candle: the features at candle *t* plus the volatility of candles *t + 1 … t + 4*.

In [2]:
conn = get_pg_connection()
with conn.cursor() as cur:
    cur.execute("SET TIME ZONE 'UTC'")   # show every timestamp from the database in UTC
conn.commit()

data = tm.load_symbols(conn, tm.SYMBOLS, task=TASK)

overview = data.groupby("symbol").agg(
    rows=("open_time", "size"), first=("open_time", "min"), last=("open_time", "max"),
    mean_vol_pct=(TARGET, lambda v: v.mean() * 100), median_vol_pct=(TARGET, lambda v: v.median() * 100),
    max_vol_pct=(TARGET, lambda v: v.max() * 100))
overview

,rows,first,last,mean_vol_pct,median_vol_pct,max_vol_pct
symbol,,,,,,
BTCUSDT,17181,2026-05-22 12:00:00+00:00,2026-11-17 11:00:00+00:00,0.15985,0.12537,2.08674
SOLUSDT,17181,2026-05-22 12:00:00+00:00,2026-11-17 11:00:00+00:00,0.25420,0.20125,4.48788


**How to read this:** the values are standard deviations of 15-minute log returns, shown in percent.
A value of 0.20 means: in that hour, a typical 15-minute move was about 0.2 % up or down. SOL usually moves more than BTC.

## 2. Look at the target

Two properties decide how we model it:

1. **It is skewed.** Volatility cannot go below zero, but occasionally it is many times its usual size.
   Its logarithm is roughly bell-shaped, which is what linear models handle well. So the models learn
   `log(volatility)`, and predictions are turned back with `exp()`.
2. **It clusters.** The volatility of the last 4 hours (feature `rv_16`) says a lot about the next hour.
   That gives a simple, honest **baseline**: "the next hour will be like the last four" (persistence).

In [3]:
shape = data.groupby("symbol")[TARGET].agg(
    skew=lambda v: v.skew(), log_skew=lambda v: np.log(v.clip(lower=tm.MIN_VOL)).skew())
cluster = data.groupby("symbol")[[tm.BASELINE_FEATURE, TARGET]].apply(lambda g: pd.Series({
    "corr(last 4h, next 1h)": g[tm.BASELINE_FEATURE].corr(g[TARGET]),
    "mean last 4h / mean next 1h": g[tm.BASELINE_FEATURE].mean() / g[TARGET].mean()}))
shape.join(cluster)

,skew,log_skew,"corr(last 4h, next 1h)",mean last 4h / mean next 1h
symbol,,,,
BTCUSDT,3.11264,-0.24612,0.42879,1.10339
SOLUSDT,3.89062,-0.11410,0.40254,1.10305


- `skew` far above 0 and `log_skew` near 0 confirms the logarithm is the better scale.
- A clearly positive correlation confirms the clustering.
- A ratio above 1 means the baseline is a bit too high on average. A standard deviation measured over only
  4 candles comes out lower than one measured over 16, so a model can gain just by correcting the level.
  That is a real improvement, but worth knowing when reading the results.

## 3. Split by time

Same rules as for the direction model: **70 % train, 15 % validation, 15 % test**, never shuffled, and the same
cut-off dates for both symbols.

**Purge:** the target of candle *t* uses candles up to *t + 4*, and the targets of neighbouring candles overlap.
So the last **4** candles before each cut are left out, otherwise training would contain prices from the validation
period.

In [4]:
val_start, test_start = tm.time_cutoffs(data)
print(f"validation from {val_start:%Y-%m-%d %H:%M} UTC, test from {test_start:%Y-%m-%d %H:%M} UTC")

rows = []
for symbol in tm.SYMBOLS:
    parts = tm.split_by_time(data[data["symbol"] == symbol], val_start, test_start, horizon=CFG["horizon"])
    for name, part in zip(("train", "validation", "test"), parts):
        rows.append({"symbol": symbol, "part": name, "rows": len(part), "from": part["open_time"].min(),
                     "to": part["open_time"].max(), "mean_vol_pct": part[TARGET].mean() * 100})
pd.DataFrame(rows).set_index(["symbol", "part"])

validation from 2026-09-24 18:30 UTC, test from 2026-10-21 14:45 UTC


rows                      from                        to  mean_vol_pct
symbol  part                                                                               
BTCUSDT train       12022 2026-05-22 12:00:00+00:00 2026-09-24 17:15:00+00:00       0.16281
        validation   2573 2026-09-24 18:30:00+00:00 2026-10-21 13:30:00+00:00       0.13177
        test         2578 2026-10-21 14:45:00+00:00 2026-11-17 11:00:00+00:00       0.17396
SOLUSDT train       12022 2026-05-22 12:00:00+00:00 2026-09-24 17:15:00+00:00       0.25532
        validation   2573 2026-09-24 18:30:00+00:00 2026-10-21 13:30:00+00:00       0.23516
        test         2578 2026-10-21 14:45:00+00:00 2026-11-17 11:00:00+00:00       0.26827

## 4. Compare models on validation

| Name | What it is |
|---|---|
| `baseline` | Persistence: predict the volatility of the last 4 hours (`rv_16`). No training |
| `ridge` | Linear regression with a penalty against large weights, on standardised features |
| `hgb_reg` | Gradient-boosted trees (`HistGradientBoostingRegressor`), kept small |

**Metrics:**
- `rmse`: typical size of the error; large misses count more. **Lower is better**, and the model is picked by it
- `mae`: average absolute error. Lower is better
- `corr`: do high predictions go with high realised volatility? 0 = no relation, 1 = perfect

As before, there is one model per symbol and one **shared** model for both (row `ALL`).

In [5]:
targets = list(tm.SYMBOLS) + [None]          # None = shared model for all symbols
built = {t: tm.build(data, t, val_start, test_start, task=TASK) for t in targets}

comparison = pd.concat({(t or "ALL"): b["comparison"] for t, b in built.items()}, names=["symbol", "model"])
comparison[["val_n", "val_rmse", "val_mae", "val_corr"]]

val_n  val_rmse  val_mae  val_corr
symbol  model                                       
BTCUSDT baseline   2573   0.00110  0.00075   0.32027
        ridge      2573   0.00092  0.00059   0.45473
        hgb_reg    2573   0.00088  0.00056   0.52826
SOLUSDT baseline   2573   0.00183  0.00128   0.29260
        ridge      2573   0.00157  0.00101   0.41961
        hgb_reg    2573   0.00154  0.00100   0.45361
ALL     baseline   5146   0.00151  0.00102   0.41022
        ridge      5146   0.00131  0.00081   0.51598
        hgb_reg    5146   0.00126  0.00078   0.56477

In [6]:
picked = pd.DataFrame({(t or "ALL"): {"picked": b["name"], "val_rmse": b["comparison"].loc[b["name"], "val_rmse"],
                                    "baseline_val_rmse": b["comparison"].loc["baseline", "val_rmse"]}
                       for t, b in built.items()}).T
picked

,picked,val_rmse,baseline_val_rmse
BTCUSDT,hgb_reg,0.00088,0.00110
SOLUSDT,hgb_reg,0.00154,0.00183
ALL,hgb_reg,0.00126,0.00151


## 5. Test the chosen models (once)

The picked model is refit on **train + validation** and scored once on the test part. The baseline is scored on the
same rows. `improvement` is how much smaller the model's RMSE is than the baseline's: 0.10 means 10 % smaller.

In [7]:
results = pd.DataFrame([tm.summary(b) for b in built.values()]).set_index("symbol")
results[["model", "val_rmse", "test_rmse", "baseline_rmse", "improvement", "test_corr", "n_test"]]

,model,val_rmse,test_rmse,baseline_rmse,improvement,test_corr,n_test
symbol,,,,,,,
BTCUSDT,hgb_reg,0.00088,0.00121,0.00146,0.17318,0.55663,2578
SOLUSDT,hgb_reg,0.00154,0.00187,0.00219,0.14826,0.50277,2578
ALL,hgb_reg,0.00126,0.00157,0.00186,0.15499,0.55884,5156


In [8]:
# the shared model, scored separately on each coin's test rows
shared = built[None]["metrics"]
pd.Series({s: shared.get(f"test_rmse_{s}") for s in tm.SYMBOLS}, name="shared model test RMSE").to_frame()

,shared model test RMSE
BTCUSDT,0.00121
SOLUSDT,0.00187


**How to read this** (write the real numbers into the report):
- A positive `improvement` means the model beats "the next hour is like the last four". Volatility is predictable
  enough that this usually works, unlike direction.
- `test_corr` well above 0 means the model separates calm from wild hours.
- Compare the per-symbol model with the shared model on each coin: the lower RMSE wins.
- Remember from step 2: part of the gain is simply a corrected level. The correlation shows whether the model also
  ranks hours better than the baseline does.

## 6. What did the models learn?

For ridge, the coefficients (on standardised features) show which features push the predicted volatility up or down.
For the tree model, permutation importance measures how much the validation error grows when one feature is shuffled.

In [9]:
from sklearn.inspection import permutation_importance


def top_features(b, n=10):
    model = b["model"]
    if b["name"] == "ridge":
        coef = pd.Series(model[-1].coef_, index=tm.FEATURE_COLS)
        return coef.reindex(coef.abs().sort_values(ascending=False).index).head(n).rename("coefficient")
    part = data if b["symbol"] is None else data[data["symbol"] == b["symbol"]]
    _, val, _ = tm.split_by_time(part, val_start, test_start, horizon=CFG["horizon"])
    log_target = np.log(val[TARGET].clip(lower=tm.MIN_VOL))     # the scale the model was trained on
    imp = permutation_importance(model, val[tm.FEATURE_COLS], log_target, scoring="neg_root_mean_squared_error",
                                 n_repeats=3, random_state=0)
    return pd.Series(imp.importances_mean, index=tm.FEATURE_COLS).sort_values(ascending=False).head(n).rename("error increase")


pd.concat({(t or "ALL"): top_features(b).rename_axis("feature").reset_index() for t, b in built.items()}, axis=1)

BTCUSDT                        SOLUSDT                              ALL               
          feature error increase         feature error increase           feature error increase
0      atr_14_rel        0.06695      atr_14_rel        0.02627        atr_14_rel        0.09744
1    hl_range_rel        0.02279        hour_cos        0.01177      hl_range_rel        0.02904
2        hour_cos        0.01564    hl_range_rel        0.01144          hour_cos        0.01350
3         dow_sin        0.01026           rv_96        0.00896             rv_96        0.01209
4       gk_var_16        0.01004       gk_var_16        0.00424           dow_sin        0.00614
5  min_to_funding        0.00622        vol_z_96        0.00414          hour_sin        0.00485
6           rv_96        0.00603  min_to_funding        0.00395         gk_var_16        0.00478
7        hour_sin        0.00578        hour_sin        0.00320    min_to_funding        0.00420
8           rv_16        0.00164         dow_sin        0.00308  log_quote_volume        0.00173
9        vol_z_96        0.00148        macd_rel        0.00267      vol_ratio_16        0.00168

Expect the volatility features on top (`rv_16`, `rv_96`, `atr_14_rel`, `gk_var_16`, `hl_range_rel`), possibly
with volume and time-of-day features after them. That would match the clustering seen in step 2.

## 7. Save, register and activate

Same mechanism as for the direction model: one artifact under `models/volatility/<symbol>/<version>.joblib`
(not committed) and one row in `model_registry` with `task = 'volatility'`. The per-symbol models are activated;
the shared model is registered for comparison.

In [10]:
ACTIVATE_PER_SYMBOL = True
ACTIVATE_SHARED = False

saved = {}
for t, b in built.items():
    activate = ACTIVATE_SHARED if t is None else ACTIVATE_PER_SYMBOL
    saved[t] = tm.persist(conn, b, activate_model=activate)

pd.DataFrame([tm.summary(b, saved[t]) for t, b in built.items()]).set_index("symbol")[
    ["model", "model_id", "model_version", "test_rmse", "improvement", "active", "artifact_path"]]

,model,model_id,model_version,test_rmse,improvement,active,artifact_path
symbol,,,,,,,
BTCUSDT,hgb_reg,10,20261117T122838Z-9ec06d3,0.00121,0.17318,True,models/volatility/BTCUSDT/20261117T122838Z-9ec...
SOLUSDT,hgb_reg,11,20261117T122838Z-9ec06d3,0.00187,0.14826,True,models/volatility/SOLUSDT/20261117T122838Z-9ec...
ALL,hgb_reg,12,20261117T122838Z-9ec06d3,0.00157,0.15499,False,models/volatility/ALL/20261117T122838Z-9ec06d3...


## 8. Write predictions for the API

`predict_all()` writes one row per symbol into `predictions`: the expected volatility of the next hour
(`y_pred_value`) and, next to it, the baseline for the same moment (`y_baseline`). The API serves it at
`GET /predict/{symbol}?task=volatility`.

In [11]:
from predict_model import predict_all  # noqa: E402

results_now, errors = predict_all(tasks=(TASK,))
pd.DataFrame(results_now)

,symbol,task,open_time,predicted_vol,model_version
0,BTCUSDT,volatility,2026-11-17T12:00:00+00:00,0.00208,20261117T122838Z-9ec06d3
1,SOLUSDT,volatility,2026-11-17T12:00:00+00:00,0.00312,20261117T122838Z-9ec06d3


**Scoring later:** one hour after a prediction, the four candles it was about have closed. The next
`build_features.py` run writes their real volatility into `outcomes`, and the view `v_prediction_quality` then
compares prediction, baseline and reality per day (`rmse` vs `rmse_baseline`). Right after training it has nothing
to score yet.

In [12]:
with conn.cursor() as cur:
    cur.execute("SELECT task, symbol, model_version, day, n_scored, rmse, rmse_baseline "
                "FROM v_prediction_quality WHERE task = 'volatility' ORDER BY day DESC, symbol LIMIT 10")
    quality = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
conn.close()
quality

,task,symbol,model_version,day,n_scored,rmse,rmse_baseline
0,volatility,BTCUSDT,20261117T122838Z-9ec06d3,2026-11-17 00:00:00+00:00,0,None,None
1,volatility,SOLUSDT,20261117T122838Z-9ec06d3,2026-11-17 00:00:00+00:00,0,None,None
